In [1]:
# ============================================================
# CELL 1 — Config
# Silver Transform: bronze_globalwatch → silver_globalwatch
# Optimizations: AQE, broadcast join, partition pruning
# ============================================================

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# Broadcast join threshold — dimensions under 50MB auto-broadcast
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(50 * 1024 * 1024))

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone

# Get DB contexts for both lakehouses
# Bronze — source
spark.sql("ADD JAR /path")  # dummy to trigger session
BRONZE_DB = spark.sql("SELECT current_database()").collect()[0][0]
print(f"Current DB: {BRONZE_DB}")

# We need to reference both lakehouses
# Fabric encodes workspace+lakehouse into the DB name
# List all available databases to find both
print("\nAvailable databases:")
spark.sql("SHOW DATABASES").show(truncate=False)

StatementMeta(, e6f73841-cb56-4fc2-a26b-d4114bfe20bf, 3, Finished, Available, Finished, False)

Current DB: chimcobldhq2aprcdth62r3nc5q66q1dchinc9bjd5m7cpbibtjmorr2c5m7eobkcdk2ap32ds

Available databases:
+--------------------------------------+
|namespace                             |
+--------------------------------------+
|globalwatch-dev.silver_globalwatch.dbo|
+--------------------------------------+



In [3]:
# ============================================================
# CELL 2 — Read Bronze + Apply Data Quality Rules
# Rules:
#   1. Drop rows with empty parameter (unclassified sensors)
#   2. Drop rows with value <= 0 or > 10000 (sensor errors)
#   3. Drop duplicate readings (same station+parameter+reading_ts)
#   4. Only keep known pollutants: pm25, pm10, no2, co, o3
# ============================================================

SILVER_DB = spark.sql("SELECT current_database()").collect()[0][0]
BRONZE_TABLE = "bronze_globalwatch.dbo.raw_openaq_readings"

# Read from Bronze
df_bronze = spark.read.format("delta").table(BRONZE_TABLE)
bronze_count = df_bronze.count()
print(f"Bronze rows read: {bronze_count}")

# --- DQ Rule 1: Drop empty parameter ---
df_dq1 = df_bronze.filter(F.col("parameter") != "")
dropped_empty_param = bronze_count - df_dq1.count()
print(f"Dropped empty parameter: {dropped_empty_param} rows")

# --- DQ Rule 2: Keep only known pollutants ---
VALID_PARAMS = ["pm25", "pm10", "no2", "co", "o3"]
df_dq2 = df_dq1.filter(F.col("parameter").isin(VALID_PARAMS))
dropped_unknown = df_dq1.count() - df_dq2.count()
print(f"Dropped unknown parameters: {dropped_unknown} rows")

# --- DQ Rule 3: Drop invalid values ---
# pm25/pm10 max realistic: 1000 µg/m³
# no2/o3 max realistic: 500 µg/m³  
# co max realistic: 50000 µg/m³
# Negative values = sensor malfunction
df_dq3 = df_dq2.filter(
    (F.col("value") > 0) & (F.col("value") < 10000)
)
dropped_invalid = df_dq2.count() - df_dq3.count()
print(f"Dropped invalid values: {dropped_invalid} rows")

# --- DQ Rule 4: Deduplicate ---
# Same station + parameter + reading timestamp = duplicate
df_dq4 = df_dq3.dropDuplicates(
    ["location_id", "parameter", "reading_ts"]
)
dropped_dupes = df_dq3.count() - df_dq4.count()
print(f"Dropped duplicates: {dropped_dupes} rows")

print(f"\nDQ Summary:")
print(f"  Bronze input : {bronze_count}")
print(f"  After DQ     : {df_dq4.count()}")
print(f"  Retention    : {round(df_dq4.count()/bronze_count*100, 1)}%")

StatementMeta(, e6f73841-cb56-4fc2-a26b-d4114bfe20bf, 7, Finished, Available, Finished, False)

Bronze rows read: 703
Dropped empty parameter: 335 rows
Dropped unknown parameters: 0 rows
Dropped invalid values: 24 rows
Dropped duplicates: 0 rows

DQ Summary:
  Bronze input : 703
  After DQ     : 344
  Retention    : 48.9%


In [4]:
# ============================================================
# CELL 3 — Enrich + AQI Category + Schema Standardization
# AQI categories based on WHO PM2.5 guidelines:
#   Good        : 0–12 µg/m³
#   Moderate    : 12.1–35.4 µg/m³
#   Unhealthy for Sensitive: 35.5–55.4
#   Unhealthy   : 55.5–150.4
#   Hazardous   : >150.4
# ============================================================

df_silver = df_dq4 \
    .withColumn(
        # AQI category — applied to pm25 only, others get 'N/A'
        "aqi_category",
        F.when(
            F.col("parameter") == "pm25",
            F.when(F.col("value") <= 12.0,   "Good")
             .when(F.col("value") <= 35.4,   "Moderate")
             .when(F.col("value") <= 55.4,   "Unhealthy for Sensitive")
             .when(F.col("value") <= 150.4,  "Unhealthy")
             .otherwise("Hazardous")
        ).otherwise("N/A")
    ) \
    .withColumn(
        # Normalize unit labels
        "unit_normalized",
        F.when(F.col("unit") == "µg/m³", "ug/m3")
         .when(F.col("unit") == "ppb",   "ppb")
         .when(F.col("unit") == "ppm",   "ppm")
         .otherwise(F.col("unit"))
    ) \
    .withColumn(
        # Round value to 2 decimal places
        "value_rounded", F.round(F.col("value"), 2)
    ) \
    .withColumn(
        # Extract date parts for reporting
        "reading_date",  F.to_date("reading_ts")
    ) \
    .withColumn(
        "reading_hour",  F.hour("reading_ts")
    ) \
    .withColumn(
        "year_month",    F.date_format("reading_ts", "yyyy-MM")
    ) \
    .withColumn(
        # Data freshness flag
        "is_recent",
        F.when(
            F.col("reading_ts") >= F.date_sub(F.current_date(), 7),
            True
        ).otherwise(False)
    ) \
    .withColumn(
        "silver_processed_ts", F.current_timestamp()
    ) \
    .select(
        # Select and order final Silver columns
        "location_id",
        "location_name",
        "city",
        "country_code",
        "country_name",
        "latitude",
        "longitude",
        "parameter",
        F.col("value_rounded").alias("value"),
        F.col("unit_normalized").alias("unit"),
        "aqi_category",
        "reading_ts",
        "reading_date",
        "reading_hour",
        "year_month",
        "is_recent",
        "ingestion_ts",
        "silver_processed_ts",
        "source_system"
    )

print(f"Silver rows: {df_silver.count()}")

# Preview AQI distribution
print("\nAQI Category distribution:")
df_silver.filter(F.col("parameter") == "pm25") \
         .groupBy("aqi_category") \
         .count() \
         .orderBy("count", ascending=False) \
         .show()

StatementMeta(, e6f73841-cb56-4fc2-a26b-d4114bfe20bf, 8, Finished, Available, Finished, False)

Silver rows: 344

AQI Category distribution:
+------------+-----+
|aqi_category|count|
+------------+-----+
|        Good|   44|
|    Moderate|   19|
|   Unhealthy|    6|
|   Hazardous|    5|
+------------+-----+



In [5]:
# ============================================================
# CELL 4 — Write to Silver Delta Table
# Partitioned by country_code + year_month
# Country-level queries are the dominant access pattern
# V-Order OFF — Silver is intermediate, not Direct Lake target
# ============================================================

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("country_code", "year_month") \
    .saveAsTable(f"{SILVER_DB}.silver_readings")

written = spark.sql(
    f"SELECT COUNT(*) as cnt FROM {SILVER_DB}.silver_readings"
).collect()[0]['cnt']

print(f"Written {written} rows → Silver ✅")

# Partition summary
print("\nPartitions created:")
spark.sql(f"""
    SELECT country_code, year_month, COUNT(*) as rows
    FROM {SILVER_DB}.silver_readings
    GROUP BY country_code, year_month
    ORDER BY rows DESC
    LIMIT 10
""").show()

StatementMeta(, e6f73841-cb56-4fc2-a26b-d4114bfe20bf, 9, Finished, Available, Finished, False)

Written 344 rows → Silver ✅

Partitions created:
+------------+----------+----+
|country_code|year_month|rows|
+------------+----------+----+
|          NL|   2016-01|  80|
|          US|   2016-01|  61|
|          CL|   2016-01|  41|
|          GB|   2016-01|  30|
|          MN|   2016-01|  23|
|          GB|   2016-02|  12|
|          IN|   2025-02|  10|
|          IN|   2016-02|  10|
|          PL|   2016-01|   9|
|          GB|   2021-04|   7|
+------------+----------+----+



In [6]:
# ============================================================
# CELL 5 — Silver Validation
# Checks: row count, null rates, value ranges, AQI distribution
# ============================================================

print("=" * 50)
print("SILVER VALIDATION REPORT")
print("=" * 50)

# Row count
total = spark.sql(
    f"SELECT COUNT(*) as cnt FROM {SILVER_DB}.silver_readings"
).collect()[0]['cnt']
print(f"\nTotal rows     : {total}")
assert total > 0, "❌ Silver table is empty!"

# Null check
print("\nNull check:")
spark.sql(f"""
    SELECT
        SUM(CASE WHEN location_id  IS NULL THEN 1 ELSE 0 END) as null_location_id,
        SUM(CASE WHEN country_code IS NULL THEN 1 ELSE 0 END) as null_country,
        SUM(CASE WHEN parameter    IS NULL THEN 1 ELSE 0 END) as null_parameter,
        SUM(CASE WHEN value        IS NULL THEN 1 ELSE 0 END) as null_value,
        SUM(CASE WHEN reading_ts   IS NULL THEN 1 ELSE 0 END) as null_reading_ts
    FROM {SILVER_DB}.silver_readings
""").show()

# Parameter distribution
print("Parameter distribution:")
spark.sql(f"""
    SELECT parameter, COUNT(*) as readings,
           ROUND(AVG(value), 2) as avg_value,
           ROUND(MAX(value), 2) as max_value
    FROM {SILVER_DB}.silver_readings
    GROUP BY parameter
    ORDER BY readings DESC
""").show()

# Country coverage
print("Country coverage:")
spark.sql(f"""
    SELECT country_code, country_name,
           COUNT(DISTINCT location_id) as stations,
           COUNT(*) as readings
    FROM {SILVER_DB}.silver_readings
    GROUP BY country_code, country_name
    ORDER BY readings DESC
    LIMIT 10
""").show(truncate=False)

# Recent data check
recent = spark.sql(f"""
    SELECT COUNT(*) as cnt
    FROM {SILVER_DB}.silver_readings
    WHERE is_recent = true
""").collect()[0]['cnt']
print(f"Recent readings (last 7 days): {recent}")

print("\n✅ Silver validation complete")
print("   Next: Run 05_gold_star_schema notebook")

StatementMeta(, e6f73841-cb56-4fc2-a26b-d4114bfe20bf, 10, Finished, Available, Finished, False)

SILVER VALIDATION REPORT

Total rows     : 344

Null check:
+----------------+------------+--------------+----------+---------------+
|null_location_id|null_country|null_parameter|null_value|null_reading_ts|
+----------------+------------+--------------+----------+---------------+
|               0|           0|             0|         0|              0|
+----------------+------------+--------------+----------+---------------+

Parameter distribution:
+---------+--------+---------+---------+
|parameter|readings|avg_value|max_value|
+---------+--------+---------+---------+
|       o3|      86|    20.54|     84.6|
|      no2|      77|    19.77|    148.0|
|     pm25|      74|    32.93|    300.0|
|     pm10|      72|    65.01|    931.0|
|       co|      35|   1183.2|   8720.0|
+---------+--------+---------+---------+

Country coverage:
+------------+--------------+--------+--------+
|country_code|country_name  |stations|readings|
+------------+--------------+--------+--------+
|NL          